# Cubic model: stationary figures

This notebook reproduces the stationary cubic-model figures from compact plot data by default. Set `USE_PRECOMPUTED_PLOT_DATA=False` and enable the simulation flags to regenerate raw data. Random seeds and all production parameters are explicit. The stationary corrections are generated symbolically from the model drift with `ucna_utils.py`.


In [ ]:
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path
import json
import pickle

import matplotlib.pyplot as plt
import numba as nb
import numpy as np
import pandas as pd
import seaborn as sns
import sympy as sp
from scipy.integrate import cumulative_trapezoid
from scipy.special import dawsn
from tqdm import tqdm
import matplotlib as mpl

plt.rcParams.update({
    'axes.labelsize': 11,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,    
})

from ucna_utils import (
    condition_on_positive_theory, histogram_density, kl_divergence, normalize_density,
    generate_stationary_correction, replica_raw_kl_uncertainty, rescaled_kl,
    simulate_endpoints,
    ucna_stationary_unnormalized, width_binned,
)

RUN_SIMULATIONS = False
RUN_INSET_SIMULATIONS = False  # expensive 7 x 7 sweep; enable explicitly
OUTPUT_DIR = Path('results/cubic_stationary')
INSET_OUTPUT_DIR = OUTPUT_DIR / 'inset_large_tau_D'
USE_PRECOMPUTED_PLOT_DATA = True
PLOT_DATA_DIR = Path('plot_data')

In [ ]:
@nb.njit
def cubic_drift(x):
    return -x**3

def cubic_drift_prime(x):
    return -3*x**2

BINNED_TAIL_TOLERANCE = 1e-8

@lru_cache(maxsize=None)
def cubic_binning_half_width(tau, D, tail_tolerance=BINNED_TAIL_TOLERANCE):
    return width_binned(
        lambda x: ucna_stationary_unnormalized(
            abs(x), tau, D, cubic_drift.py_func, cubic_drift_prime,
            lower_bound=0.0,
        ),
        tail_tolerance=tail_tolerance,
    )

## Production configurations and simulation


In [ ]:
@dataclass(frozen=True)
class CubicExperiment:
    tau: float
    D: float
    dt: float = 0.01
    duration: float = 10.0
    n_replicas: int = 10
    samples_per_replica: int = 100000
    n_bins: int = 200
    # Multiplier of the tail-controlled half-width from cubic_binning_half_width.
    domain_widths: float = 1.0
    base_seed: int = 20260809

    @property
    def n_steps(self):
        steps = int(round(self.duration / self.dt))
        if not np.isclose(steps * self.dt, self.duration):
            raise ValueError('duration must be an integer multiple of dt')
        return steps

    @property
    def total_samples(self):
        return self.n_replicas * self.samples_per_replica

# Parameter pairs used for the main stationary sweep in cubic_clean.ipynb.
TAUS = [0.1, 0.2, 0.3, 1.0, 2.0, 3.0]
DS_BY_TAU = [
    [0.1,0.2,0.3,1, 2, 3, 10, 20], [0.1,0.2,0.3, 1, 2, 3, 10], [0.1,0.2, 0.3, 1, 2, 3],#[1, 2, 3, 10, 20], [0.3, 1, 2, 3, 10], [0.2, 0.3, 1, 2, 3],
    [0.1, 0.2, 0.3, 1, 2], [0.03, 0.1, 0.2, 0.3, 1],
    [0.02, 0.03, 0.1, 0.2, 0.3],
]
BASE_PRODUCTION_CONFIGS = [
    CubicExperiment(tau=tau, D=D)
    for tau, values in zip(TAUS, DS_BY_TAU) for D in values
]
# Add new parameter pairs here; saved base results will not be rerun.
ADDITIONAL_PRODUCTION_CONFIGS = []
ALL_PRODUCTION_CONFIGS = BASE_PRODUCTION_CONFIGS + ADDITIONAL_PRODUCTION_CONFIGS
pd.DataFrame(asdict(config) for config in ALL_PRODUCTION_CONFIGS).head()

config_table = pd.DataFrame(asdict(config) for config in ALL_PRODUCTION_CONFIGS)
config_table['trajectory_steps_total'] = (
    config_table['n_replicas']
    * config_table['samples_per_replica']
    * (config_table['duration'] / config_table['dt']).astype(int)
)
#display(inset_config_table)
print(f'{len(ALL_PRODUCTION_CONFIGS)} configurations; '
      f'{config_table.trajectory_steps_total.sum():,.0f} total trajectory steps')

# Inset

INSET_TAUS = 10.0 ** np.arange(-1, 6)
INSET_DS = 10.0 ** np.arange(-1, 6)
INSET_REPLICAS = 5
INSET_SAMPLES_PER_REPLICA = 10_000

def inset_duration(tau, D, dt=0.01):
    requested = max(5 * np.sqrt(tau) / D**0.25, 10.0)
    return max(1, int(requested // dt)) * dt

INSET_CONFIGS = [
    CubicExperiment(
        tau=float(tau), D=float(D), duration=inset_duration(tau, D),
        n_replicas=INSET_REPLICAS,
        samples_per_replica=INSET_SAMPLES_PER_REPLICA,
    )
    for tau in INSET_TAUS for D in INSET_DS
]
inset_config_table = pd.DataFrame(asdict(config) for config in INSET_CONFIGS)
inset_config_table['trajectory_steps_total'] = (
    inset_config_table['n_replicas']
    * inset_config_table['samples_per_replica']
    * (inset_config_table['duration'] / inset_config_table['dt']).astype(int)
)
#display(inset_config_table)
print(f'{len(INSET_CONFIGS)} inset configurations; '
      f'{inset_config_table.trajectory_steps_total.sum():,.0f} total trajectory steps')

In [ ]:
def replica_seed(config, replica):
    tau_key = int(round(config.tau * 1_000_000))
    D_key = int(round(config.D * 1_000_000))
    sequence = np.random.SeedSequence([config.base_seed, tau_key, D_key, replica])
    return int(sequence.generate_state(1, dtype=np.uint32)[0])

def bin_edges_for(config):
    half_width = (
        config.domain_widths * cubic_binning_half_width(config.tau, config.D)
    )
    return np.linspace(-half_width, half_width, config.n_bins + 1)

def run_stationary_config(config):
    edges = bin_edges_for(config)
    replica_endpoints = []
    replica_counts = []
    replica_seeds = []
    outside_counts = []
    replicas = tqdm(
        range(config.n_replicas),
        desc=f'replicas: tau={config.tau:g}, D={config.D:g}',
        leave=False,
    )
    for replica in replicas:
        seed = replica_seed(config, replica)
        x_final, _ = simulate_endpoints(
            cubic_drift, config.D, config.tau, config.dt, config.n_steps,
            config.samples_per_replica, seed,
        )
        counts, _ = np.histogram(x_final, bins=edges)
        replica_endpoints.append(x_final)
        replica_counts.append(counts)
        replica_seeds.append(seed)
        outside_counts.append(int(len(x_final) - counts.sum()))
    return {
        'schema_version': 3,
        'config': asdict(config), 'edges': edges,
        'replica_endpoints': np.asarray(replica_endpoints),
        'replica_counts': np.asarray(replica_counts),
        'replica_seeds': np.asarray(replica_seeds),
        'replica_seed_chunks': [[int(seed)] for seed in replica_seeds],
        'replica_chunk_sizes': [[config.samples_per_replica] for _ in replica_seeds],
        'outside_counts': np.asarray(outside_counts),
    }

def rebin_result(result, n_bins=None, domain_widths=None, half_width=None):
    # Reuses saved endpoints; no stochastic dynamics are rerun.
    if 'replica_endpoints' not in result:
        raise ValueError('This result predates endpoint storage and must be rerun')
    config_values = dict(result['config'])
    if n_bins is not None:
        config_values['n_bins'] = int(n_bins)
    if domain_widths is not None:
        config_values['domain_widths'] = float(domain_widths)
    if half_width is not None:
        if domain_widths is not None:
            raise ValueError('Specify either domain_widths or half_width, not both')
        config_values['domain_widths'] = float(half_width) / cubic_binning_half_width(
            config_values['tau'], config_values['D']
        )
    config = CubicExperiment(**config_values)
    edges = bin_edges_for(config)
    endpoints = np.asarray(result['replica_endpoints'])
    counts = np.asarray([
        np.histogram(values, bins=edges)[0] for values in endpoints
    ])
    outside = np.asarray([
        len(values) - row.sum() for values, row in zip(endpoints, counts)
    ])
    return {
        'schema_version': result.get('schema_version', 2),
        'config': asdict(config), 'edges': edges,
        'replica_endpoints': endpoints,
        'replica_counts': counts,
        'replica_seeds': np.asarray(result['replica_seeds']),
        **({
            'replica_seed_chunks': result['replica_seed_chunks'],
            'replica_chunk_sizes': result['replica_chunk_sizes'],
        } if 'replica_seed_chunks' in result else {}),
        'outside_counts': outside,
    }

def load_saved_stationary_data(output_dir=OUTPUT_DIR):
    loaded = {}
    for filename in sorted(output_dir.glob('stationary_tau_*_D_*.pkl')):
        with filename.open('rb') as handle:
            result = pickle.load(handle)
        if 'replica_endpoints' not in result:
            print(f'Skipping stale result until it is rerun: {filename}')
            continue
        config = result['config']
        key = (config['tau'], config['D'])
        if key in loaded:
            raise ValueError(f'Duplicate saved result for tau={key[0]}, D={key[1]}')
        loaded[key] = result
    return loaded

def stationary_filename(config, output_dir=OUTPUT_DIR):
    return output_dir / f'stationary_tau_{config.tau:g}_D_{config.D:g}.pkl'

def supplemental_seed(config, replica, previous_sample_count):
    # A new deterministic stream: it never repeats the seed of the saved chunk.
    tau_key = int(round(config.tau * 1_000_000))
    D_key = int(round(config.D * 1_000_000))
    sequence = np.random.SeedSequence([
        config.base_seed, tau_key, D_key, replica,
        0xA5A5A5A5, int(previous_sample_count),
    ])
    return int(sequence.generate_state(1, dtype=np.uint32)[0])

def extend_stationary_result(saved, target_config):
    old_config = dict(saved['config'])
    target_values = asdict(target_config)
    scalable = {'n_replicas', 'samples_per_replica'}
    changed_fixed = {
        key: (old_config.get(key), target_values[key])
        for key in target_values
        if key not in scalable and old_config.get(key) != target_values[key]
    }
    if changed_fixed:
        raise ValueError(f'Non-sample configuration changes cannot be appended: {changed_fixed}')
    old_replicas = int(old_config['n_replicas'])
    old_samples = int(old_config['samples_per_replica'])
    if target_config.n_replicas < old_replicas:
        raise ValueError('n_replicas cannot be reduced in an existing result')
    if target_config.samples_per_replica < old_samples:
        raise ValueError('samples_per_replica cannot be reduced in an existing result')

    endpoints = [np.asarray(row) for row in saved['replica_endpoints']]
    primary_seeds = [int(seed) for seed in saved['replica_seeds']]
    if 'replica_seed_chunks' in saved:
        seed_chunks = [list(map(int, row)) for row in saved['replica_seed_chunks']]
        chunk_sizes = [list(map(int, row)) for row in saved['replica_chunk_sizes']]
    else:
        seed_chunks = [[seed] for seed in primary_seeds]
        chunk_sizes = [[old_samples] for _ in range(old_replicas)]

    work = []
    if target_config.samples_per_replica > old_samples:
        work.extend((replica, target_config.samples_per_replica - old_samples)
                    for replica in range(old_replicas))
    work.extend((replica, target_config.samples_per_replica)
                for replica in range(old_replicas, target_config.n_replicas))

    for replica, n_new in tqdm(
        work, desc=f'extending tau={target_config.tau:g}, D={target_config.D:g}',
        leave=False,
    ):
        if replica < old_replicas:
            previous_count = len(endpoints[replica])
            seed = supplemental_seed(target_config, replica, previous_count)
        else:
            seed = replica_seed(target_config, replica)
        new_endpoints, _ = simulate_endpoints(
            cubic_drift, target_config.D, target_config.tau, target_config.dt,
            target_config.n_steps, n_new, seed,
        )
        if replica < old_replicas:
            endpoints[replica] = np.concatenate([endpoints[replica], new_endpoints])
            seed_chunks[replica].append(seed)
            chunk_sizes[replica].append(n_new)
        else:
            endpoints.append(new_endpoints)
            primary_seeds.append(seed)
            seed_chunks.append([seed])
            chunk_sizes.append([n_new])

    endpoints = np.asarray(endpoints)
    edges = bin_edges_for(target_config)
    counts = np.asarray([np.histogram(row, bins=edges)[0] for row in endpoints])
    outside = np.asarray([len(row) - count.sum() for row, count in zip(endpoints, counts)])
    return {
        'schema_version': 3, 'config': target_values, 'edges': edges,
        'replica_endpoints': endpoints, 'replica_counts': counts,
        'replica_seeds': np.asarray(primary_seeds),
        'replica_seed_chunks': seed_chunks, 'replica_chunk_sizes': chunk_sizes,
        'outside_counts': outside,
    }

def run_missing_stationary_configs(configs, output_dir=OUTPUT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    actions = []
    reusable = 0
    for config in tqdm(configs, desc='checking saved configurations'):
        filename = stationary_filename(config, output_dir)
        if filename.exists():
            with filename.open('rb') as handle:
                saved = pickle.load(handle)
            if 'replica_endpoints' not in saved:
                actions.append(('rerun', config, filename, None))
            elif saved.get('config') == asdict(config):
                reusable += 1
            else:
                # Histogram-only changes reuse endpoints and never rerun dynamics.
                saved_config = dict(saved['config'])
                target_values = asdict(config)
                fixed = set(target_values) - {'n_replicas', 'samples_per_replica'}
                differences = {key for key in fixed
                               if saved_config.get(key) != target_values[key]}
                histogram_fields = {'n_bins', 'domain_widths'}
                non_histogram_differences = differences - histogram_fields
                decreases = (
                    target_values['n_replicas'] < saved['config']['n_replicas']
                    or target_values['samples_per_replica'] < saved['config']['samples_per_replica']
                )
                if non_histogram_differences or decreases:
                    raise ValueError(
                        f'{filename} differs in non-appendable settings: '
                        f'fields={sorted(non_histogram_differences)}, decrease={decreases}'
                    )
                sample_increase = (
                    target_values['n_replicas'] > saved['config']['n_replicas']
                    or target_values['samples_per_replica'] > saved['config']['samples_per_replica']
                )
                kind = 'extend' if sample_increase else 'rebin'
                actions.append((kind, config, filename, saved))
        else:
            actions.append(('new', config, filename, None))
    counts = {kind: sum(action[0] == kind for action in actions)
              for kind in ['new', 'rerun', 'rebin', 'extend']}
    print(f"{reusable} reusable, {counts['extend']} extendable, "
          f"{counts['rebin']} rebin-only, {counts['new']} new, "
          f"{counts['rerun']} stale")
    for kind, config, filename, saved in tqdm(
        actions, desc='stationary configurations'
    ):
        if kind in {'extend', 'rebin'}:
            print(f'{kind}: tau={config.tau:g}, D={config.D:g}')
            saved = rebin_result(
                saved, n_bins=config.n_bins,
                domain_widths=config.domain_widths,
            )
        if kind == 'extend':
            result = extend_stationary_result(saved, config)
        elif kind == 'rebin':
            result = saved
        else:
            result = run_stationary_config(config)
        temporary = filename.with_suffix('.pkl.tmp')
        with temporary.open('wb') as handle:
            pickle.dump(result, handle)
        temporary.replace(filename)

In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Using packaged cubic plot data; raw results are not loaded.')
else:
    if RUN_SIMULATIONS:
        run_missing_stationary_configs(ALL_PRODUCTION_CONFIGS)

    # Always analyze the union of every compatible result already saved.
    stationary_data = load_saved_stationary_data()
    print(f'Loaded {len(stationary_data)} stationary parameter pairs')


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Using the packaged cubic inset curve.')
else:
    # This flag is separate so rerunning the main notebook cannot start the costly inset sweep.
    if RUN_INSET_SIMULATIONS:
        run_missing_stationary_configs(INSET_CONFIGS, output_dir=INSET_OUTPUT_DIR)

    inset_stationary_data = load_saved_stationary_data(INSET_OUTPUT_DIR)
    expected_inset_keys = {(config.tau, config.D) for config in INSET_CONFIGS}
    missing_inset_keys = sorted(expected_inset_keys - set(inset_stationary_data))
    print(f'Loaded {len(inset_stationary_data)}/{len(INSET_CONFIGS)} inset parameter pairs')
    if missing_inset_keys:
        print(f'{len(missing_inset_keys)} inset pairs remain; set RUN_INSET_SIMULATIONS=True to run them')


## Theoretical bin probabilities


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Symbolic corrections are generated only in raw-analysis mode.')
else:
    def ucna_bin_probabilities(tau, D, edges):
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        density = np.array([
            ucna_stationary_unnormalized(
                x, tau, D, cubic_drift.py_func, cubic_drift_prime,
                lower_bound=edges[0],
            )
            for x in centers
        ])
        weights = density * widths
        if np.any(weights <= 0) or not np.all(np.isfinite(weights)):
            raise ValueError('UCNA produced nonpositive or nonfinite bin weights')
        return weights / weights.sum()

    def cbfpe_effective_diffusion(x, tau, D):
        x = np.asarray(x, dtype=float)
        z = np.empty_like(x)
        nonzero = x != 0
        z[nonzero] = 1.0 / (np.sqrt(2.0 * tau) * x[nonzero])
        product = np.zeros_like(x)
        product[nonzero] = (
            np.sqrt(2.0 * tau) * x[nonzero] * dawsn(z[nonzero])
        )
        return D * (1.0 + 3.0 * tau * x**2 * (product - 1.0))

    def cbfpe_bin_probabilities(tau, D, edges):
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        diffusion = cbfpe_effective_diffusion(centers, tau, D)
        if np.any(~np.isfinite(diffusion)) or np.any(diffusion <= 0):
            raise ValueError('cBFPE effective diffusion is nonpositive or nonfinite')
        integrand = cubic_drift.py_func(centers) / diffusion
        exponent = cumulative_trapezoid(integrand, centers, initial=0.0)
        log_weights = exponent - np.log(diffusion) + np.log(widths)
        log_weights -= log_weights.max()
        weights = np.exp(log_weights)
        if np.any(~np.isfinite(weights)) or weights.sum() <= 0:
            raise ValueError('cBFPE produced invalid bin weights')
        return weights / weights.sum()

    CORRECTION_VARIABLE = sp.symbols('x', real=True)

    @lru_cache(maxsize=None)
    def generated_cubic_correction(tau, D):
        return generate_stationary_correction(
            -CORRECTION_VARIABLE**3, CORRECTION_VARIABLE, float(tau), float(D)
        )

    # Generate corrections
    MAIN_CORRECTION_KEYS = sorted(
        (float(tau), float(D)) for tau, D in stationary_data
    )
    CORRECTIONS = {
        key: generated_cubic_correction(*key)
        for key in tqdm(MAIN_CORRECTION_KEYS, desc='Generating symbolic corrections')
    }
    print(f'Generated {len(CORRECTIONS)} main-sweep corrections')

    NEGATIVE_MASS_THRESHOLD = 1e-8
    EXCLUDED_EMPIRICAL_MASS_THRESHOLD = 1e-3

    def corrected_bin_weights(correction, edges):
        centers = 0.5 * (edges[:-1] + edges[1:])
        weights = np.asarray(correction(centers), dtype=float) * np.diff(edges)
        if np.any(~np.isfinite(weights)) or weights.sum() <= 0:
            raise ValueError('Corrected solution has nonfinite or nonpositive total weight')
        normalized = weights / weights.sum()
        negative_mass = float(-normalized[normalized < 0].sum())
        return normalized, negative_mass


In [ ]:
def summarize_result(result):
    config = result['config']
    theory = ucna_bin_probabilities(config['tau'], config['D'], result['edges'])
    counts_by_replica = result['replica_counts']
    replica_raw_kl = np.array([
        kl_divergence(counts, theory)
        for counts in counts_by_replica
        ])
    pooled = counts_by_replica.sum(axis=0)
    n_total = int(config['n_replicas'] * config['samples_per_replica'])
    pooled_scale = n_total / (len(pooled) - 1)
    pooled_dkl = pooled_scale * kl_divergence(pooled, theory)
    replica_raw_sem = replica_raw_kl.std(ddof=1) / np.sqrt(len(replica_raw_kl))
    return {
        'tau': config['tau'], 'D': config['D'],
        'D_tau2': config['D'] * config['tau']**2,
        'n_replicas': config['n_replicas'],
        'samples_total': n_total,
        'samples_in_range': int(pooled.sum()),
        'outside_total': int(result['outside_counts'].sum()),
        'dkl_pooled': pooled_dkl,
        'replica_raw_KL_mean': replica_raw_kl.mean(),
        'replica_raw_KL_sem': replica_raw_sem,
        'dkl_uncertainty': pooled_scale * replica_raw_sem,
    }

## Stationary-density comparison


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    FIGURE_2A_CONFIGS = [(0.2, 1.0), (0.2, 2.0)]
    FIGURE_2A_X_MIN = -2.0
    FIGURE_2A_X_MAX = 2.0
    print('Using packaged cubic density data.')
else:
    # This is the desired TOTAL number of endpoints in Fig. 2a, including the
    # production endpoints. Only a shortfall is simulated and stored separately.
    FIGURE_2A_CONFIGS = [(0.2, 1.0), (0.2, 2.0)]
    FIGURE_2A_OUTPUT_DIR = OUTPUT_DIR / 'figure_2a'
    FIGURE_2A_TARGET_TOTAL_SAMPLES = 10000000
    RUN_FIGURE_2A_SIMULATIONS = False
    FIGURE_2A_X_MIN = -2.0
    FIGURE_2A_X_MAX = 2.0
    FIGURE_2A_NBINS = 60  # bins across [FIGURE_2A_X_MIN, FIGURE_2A_X_MAX]

    figure_2a_data = load_saved_stationary_data(FIGURE_2A_OUTPUT_DIR)
    figure_2a_add_on_samples = {}
    figure_2a_configs = []
    for tau, D in FIGURE_2A_CONFIGS:
        key = (tau, D)
        production_samples = np.asarray(
            stationary_data[key]['replica_endpoints']
        ).size
        required_add_on = max(
            0, FIGURE_2A_TARGET_TOTAL_SAMPLES - production_samples
        )
        figure_2a_add_on_samples[key] = required_add_on
        saved_add_on = (
            np.asarray(figure_2a_data[key]['replica_endpoints']).size
            if key in figure_2a_data else 0
        )
        print(
            f'tau={tau:g}, D={D:g}: target={FIGURE_2A_TARGET_TOTAL_SAMPLES:,}, '
            f'production={production_samples:,}, required add-on={required_add_on:,}, '
            f'saved add-on={saved_add_on:,}'
        )
        if saved_add_on >= required_add_on:
            continue
        config_values = dict(stationary_data[(tau, D)]['config'])
        config_values['n_replicas'] = 1
        config_values['samples_per_replica'] = required_add_on
        config_values['n_bins'] = FIGURE_2A_NBINS
        # Keep the dedicated ensemble independent of the production seed stream.
        config_values['base_seed'] += 1
        figure_2a_configs.append(CubicExperiment(**config_values))

    if figure_2a_configs:
        display(pd.DataFrame(asdict(config) for config in figure_2a_configs))
    if RUN_FIGURE_2A_SIMULATIONS and figure_2a_configs:
        run_missing_stationary_configs(figure_2a_configs, FIGURE_2A_OUTPUT_DIR)
    figure_2a_data = load_saved_stationary_data(FIGURE_2A_OUTPUT_DIR)
    missing = [
        key for key, required in figure_2a_add_on_samples.items()
        if required > 0 and (
            key not in figure_2a_data
            or np.asarray(figure_2a_data[key]['replica_endpoints']).size < required
        )
    ]
    print(
        f'{len(figure_2a_data)} dedicated Fig. 2a configurations loaded; '
        f'{len(missing)} remain to be simulated.'
    )
    if missing and not RUN_FIGURE_2A_SIMULATIONS:
        print('Set RUN_FIGURE_2A_SIMULATIONS=True to run them.')


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    density_plot_data = pd.read_csv(
        PLOT_DATA_DIR / 'cubic_density.csv'
    )

    fig, axes = plt.subplots(
        ncols=1, nrows=3, sharex='col', sharey='row', figsize=(4, 4.5),
        height_ratios=[8, 3, 3], constrained_layout=True,
    )
    ax_main = axes[0]
    colors = ['C0', 'C1']

    for deviation_ax, key, color in zip(axes[1:], FIGURE_2A_CONFIGS, colors):
        tau, D = key
        data = density_plot_data[
            np.isclose(density_plot_data['tau'], tau)
            & np.isclose(density_plot_data['D'], D)
        ].sort_values('x')
        centers = data['x'].to_numpy()
        empirical = data['simulation'].to_numpy()
        empirical_sem = data['simulation_uncertainty'].to_numpy()
        ucna = data['UCNA'].to_numpy()
        corrected = data['Correction'].to_numpy()
        cbfpe = data['cBFPE'].to_numpy()
        label = rf'$\tau = {tau:g}, D={D:g}$'

        ax_main.errorbar(
            centers, empirical, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8,
        )
        ax_main.plot(
            centers, empirical, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            label=label, zorder=9,
        )
        ax_main.plot(centers, ucna, alpha=0.8, c=color, label='UCNA, LLA')
        ax_main.plot(centers, corrected, alpha=1, c=color,
                     label='Correction', linestyle='--')
        ax_main.plot(centers, cbfpe, alpha=0.8, c=color,
                     label='cBFPE', linestyle=':')

        deviation = empirical - ucna
        deviation_ax.plot(centers, centers * 0.0, alpha=0.5, c=color,
                          label='UCNA, LLA')
        deviation_ax.errorbar(
            centers, deviation, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8, alpha=0.7,
        )
        deviation_ax.plot(
            centers, deviation, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            label=label, zorder=9, alpha=0.7,
        )
        deviation_ax.plot(centers, cbfpe - ucna, alpha=1, c=color,
                          linestyle=':', label='cBFPE')
        deviation_ax.plot(centers, corrected - ucna, alpha=1, c=color,
                          linestyle='--', label='Correction')

    ax_main.set_ylabel('Probability Density')
    axes[1].set_ylabel('Deviation from UCNA', loc='top')
    axes[2].set_ylabel('')
    axes[2].set_xlabel('x')

    legend_handles = [
        mpl.lines.Line2D([], [], color='black', ls='-', label='UCNA, LLA'),
        mpl.lines.Line2D([], [], color='black', ls='--', label='Correction'),
        mpl.lines.Line2D([], [], color='black', ls=':', label='cBFPE'),
    ]
    for tau, D, color in zip([0.2, 0.2], [1, 2], ['C0', 'C1']):
        legend_handles.append(ax_main.errorbar(
            [np.nan], [np.nan], yerr=[0.1], fmt='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            ecolor=color, elinewidth=1.0, capsize=2.5, capthick=1.0,
            label=fr'$\tau = {tau:g}, D = {D:g}$',
        ))
    ax_main.legend(handles=legend_handles)
    axes[-1].set_xlim(FIGURE_2A_X_MIN, FIGURE_2A_X_MAX)
    axes[1].set_yticks([0.00, 0.02])
    axes[2].set_yticks([0.00, 0.02])
    axes[0].annotate(r'$\left(a\right)$', (0.01, 0.9), xycoords='axes fraction',
                     fontsize=15, fontweight='bold')
    plt.show()
else:
    # Figure 2a: error bars are drawn below open simulation markers.
    missing = [
        key for key, required in figure_2a_add_on_samples.items()
        if required > 0 and key not in figure_2a_data
    ]
    if missing:
        raise RuntimeError(f'Figure 2a configurations are not loaded: {missing}')

    fig, axes = plt.subplots(
        ncols=1, nrows=3, sharex='col', sharey='row', figsize=(4, 4.5),
        height_ratios=[8, 3, 3], constrained_layout=True,
    )
    ax_main = axes[0]
    colors = ['C0', 'C1']

    for deviation_ax, key, color in zip(axes[1:], FIGURE_2A_CONFIGS, colors):
        result = stationary_data[key]
        tau, D = key
        edges = np.linspace(
            FIGURE_2A_X_MIN, FIGURE_2A_X_MAX, FIGURE_2A_NBINS + 1
        )
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        production_endpoints = np.asarray(
            result['replica_endpoints']
        ).reshape(-1)[:FIGURE_2A_TARGET_TOTAL_SAMPLES]
        production_counts = np.histogram(production_endpoints, bins=edges)[0]
        required_add_on = figure_2a_add_on_samples[key]
        if required_add_on:
            add_on_endpoints = np.asarray(
                figure_2a_data[key]['replica_endpoints']
            ).reshape(-1)[:required_add_on]
            add_on_counts = np.histogram(add_on_endpoints, bins=edges)[0]
        else:
            add_on_counts = np.zeros_like(production_counts)
        counts = production_counts + add_on_counts
        n_in_range = counts.sum()
        empirical = counts / (n_in_range * widths)
        empirical_sem = np.sqrt(counts) / (n_in_range * widths)

        ucna = ucna_bin_probabilities(tau, D, edges) / widths
        cbfpe = cbfpe_bin_probabilities(tau, D, edges) / widths
        corrected_weights, negative_mass = corrected_bin_weights(CORRECTIONS[key], edges)
        corrected = corrected_weights / widths
        label = rf'$\tau = {tau:g}, D={D:g}$'

        # Error bars first, then smaller open squares above them.
        ax_main.errorbar(
            centers, empirical, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8,
        )
        ax_main.plot(
            centers, empirical, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            label=label, zorder=9,
        )
        ax_main.plot(centers, ucna, alpha=0.8, c=color, label='UCNA, LLA')
        ax_main.plot(centers, corrected, alpha=1, c=color, label='Correction', linestyle='--')
        ax_main.plot(centers, cbfpe, alpha=0.8, c=color, label='cBFPE', linestyle=':')

        deviation = empirical - ucna
        deviation_ax.plot(centers, centers * 0.0, alpha=0.5, c=color, label='UCNA, LLA')
        #deviation_ax.plot(centers, centers * 0.0, alpha=0.5, c='black', label='UCNA, LLA')
        deviation_ax.errorbar(
            centers, deviation, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8, alpha = 0.7
        )
        deviation_ax.plot(
            centers, deviation, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            label=label, zorder=9, alpha = 0.7 
        )
        deviation_ax.plot(centers, cbfpe - ucna, alpha=1, c=color, linestyle=':', label='cBFPE')
        #deviation_ax.plot(centers, cbfpe - ucna, alpha=1, c=color, linestyle=':', label='cBFPE')
        deviation_ax.plot(centers, corrected - ucna, alpha=1, c=color, linestyle='--', label='Correction')
        #deviation_ax.plot(centers, corrected - ucna, alpha=1, c=color, linestyle='--', label='Correction')
        if negative_mass > NEGATIVE_MASS_THRESHOLD:
            print(f'Warning: correction at {key} has negative mass {negative_mass:.3e}')

    ax_main.set_ylabel('Probability Density')
    axes[1].set_ylabel('Deviation from UCNA', loc='top')
    axes[2].set_ylabel('')
    axes[2].set_xlabel('x')

    legend_handles = [
        mpl.lines.Line2D([], [], color='black', ls='-', label='UCNA, LLA'),
        mpl.lines.Line2D([], [], color='black', ls='--', label='Correction'),
        mpl.lines.Line2D([], [], color='black', ls=':', label='cBFPE'),
    ]
    for tau, D, color in zip([0.2,0.2],[1,2],['C0','C1']):
        legend_handles.append(axes[0].errorbar(
            [np.nan], [np.nan], yerr=[0.1], fmt='s', markersize=2.2, 
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            ecolor=color, elinewidth=1.0, capsize=2.5, capthick=1.0,
            label=fr'$\tau = {tau:g}, D = {D:g}$'))
    ax_main.legend(handles = legend_handles)
    axes[-1].set_xlim(FIGURE_2A_X_MIN, FIGURE_2A_X_MAX)
    axes[1].set_yticks([0.00, 0.02])
    axes[2].set_yticks([0.00, 0.02])

    axes[0].annotate(r'$\left(a\right)$', (0.01, 0.9), xycoords='axes fraction',
                 fontsize=15, fontweight='bold')

    plt.show()


## KL-divergence figure


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    figure_2b_results = pd.read_csv(PLOT_DATA_DIR / 'cubic_kl.csv')
else:
    figure_2b_rows = []
    for (tau, D), result in tqdm(stationary_data.items(), desc='Figure 2b KL values'):
        edges = result['edges']
        counts_by_replica = np.asarray(result['replica_counts'])
        counts = counts_by_replica.sum(axis=0)
        n_in_range = int(counts.sum())
        n_total = int(
            result['config']['n_replicas'] * result['config']['samples_per_replica']
        )
        outside = int(result['outside_counts'].sum())
        ucna = ucna_bin_probabilities(tau, D, edges)
        ucna_raw_mean, ucna_raw_sem, ucna_uncertainty = replica_raw_kl_uncertainty(
            counts_by_replica, ucna, n_total
        )
        figure_2b_rows.append({
            'tau': tau, 'D': D, 'N': n_total, 'Nbins': len(counts),
            'Approximation': 'UCNA',
            'KL': n_total / (len(counts) - 1) * kl_divergence(counts, ucna),
            'replica_raw_KL_mean': ucna_raw_mean,
            'replica_raw_KL_sem': ucna_raw_sem,
            'KL_uncertainty': ucna_uncertainty,
            'outside': outside,
        })
        cbfpe = cbfpe_bin_probabilities(tau, D, edges)
        cbfpe_raw_mean, cbfpe_raw_sem, cbfpe_uncertainty = (
            replica_raw_kl_uncertainty(counts_by_replica, cbfpe, n_total)
        )
        figure_2b_rows.append({
            'tau': tau, 'D': D, 'N': n_total, 'Nbins': len(counts),
            'Approximation': 'cBFPE',
            'KL': n_total / (len(counts) - 1) * kl_divergence(counts, cbfpe),
            'replica_raw_KL_mean': cbfpe_raw_mean,
            'replica_raw_KL_sem': cbfpe_raw_sem,
            'KL_uncertainty': cbfpe_uncertainty,
            'outside': outside,
        })
        correction = CORRECTIONS.get((float(tau), float(D)))
        if correction is not None:
            corrected, negative_mass = corrected_bin_weights(correction, edges)
            conditioned_counts, conditioned_correction, excluded_fraction = (
                condition_on_positive_theory(counts_by_replica, corrected)
            )
            retained_samples = int(conditioned_counts.sum())
            correction_valid = (negative_mass <= NEGATIVE_MASS_THRESHOLD
                                and excluded_fraction <= EXCLUDED_EMPIRICAL_MASS_THRESHOLD)
            conditional_dkl = retained_samples / (len(conditioned_correction) - 1) * kl_divergence(
                conditioned_counts.sum(axis=0), conditioned_correction
            )
            if correction_valid:
                corrected_raw_mean, corrected_raw_sem, corrected_uncertainty = (
                    replica_raw_kl_uncertainty(
                        conditioned_counts, conditioned_correction, retained_samples
                    )
                )
                corrected_kl = conditional_dkl
            else:
                corrected_raw_mean = np.nan
                corrected_raw_sem = np.nan
                corrected_uncertainty = np.nan
                corrected_kl = conditional_dkl
            figure_2b_rows.append({
                'tau': tau, 'D': D, 'N': n_total, 'Nbins': len(counts),
                'Approximation': 'Correction',
                'KL': corrected_kl,
                'replica_raw_KL_mean': corrected_raw_mean,
                'replica_raw_KL_sem': corrected_raw_sem,
                'KL_uncertainty': corrected_uncertainty,
                'outside': outside,
                'nonpositive_theory_bins': int(np.sum(corrected <= 0)),
                'negative_mass': negative_mass,
                'excluded_empirical_fraction': excluded_fraction,
                'correction_valid': correction_valid,
            })

    figure_2b_results = pd.DataFrame(figure_2b_rows)
    figure_2b_results['nonpositive_theory_bins'] = (
        figure_2b_results['nonpositive_theory_bins'].fillna(0).astype(int)
    )
    figure_2b_results['negative_mass'] = figure_2b_results['negative_mass'].fillna(0.0)
    figure_2b_results['correction_valid'] = (
        figure_2b_results['correction_valid'].fillna(True).astype(bool)
    )
    figure_2b_results['tau2D'] = figure_2b_results['tau']**2 * figure_2b_results['D']
    if figure_2b_results['outside'].sum() > 0:
        print('Warning: KL uses total generated N in the prefactor while the histogram is normalized over in-range samples.')
    invalid_corrections = figure_2b_results[
        (figure_2b_results['Approximation'] == 'Correction')
        & (~figure_2b_results['correction_valid'])
    ]
    if not invalid_corrections.empty:
        print(f'{len(invalid_corrections)} corrections exceed a mass threshold; their conditional KL values are shown without uncertainty bands.')
    figure_2b_results.sort_values(['Approximation', 'tau', 'D'])


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    inset_results = pd.read_csv(PLOT_DATA_DIR / 'cubic_inset_pairs.csv')
    inset_curve = pd.read_csv(PLOT_DATA_DIR / 'cubic_inset_curve.csv')
    inset_exclusions = pd.DataFrame()
    inset_warnings = pd.DataFrame()
    missing_inset_keys = []
else:
    inset_rows = []
    inset_exclusions = []
    inset_warnings = []
    for (tau, D), result in tqdm(
        inset_stationary_data.items(), desc='Figure 2b inset KL values'
    ):
        endpoints = np.asarray(result.get('replica_endpoints', []))
        counts = np.asarray(result.get('replica_counts', []))
        reasons = []
        if endpoints.size == 0:
            reasons.append('missing endpoints')
        else:
            n_nonfinite = int(np.size(endpoints) - np.isfinite(endpoints).sum())
            if n_nonfinite == endpoints.size:
                reasons.append(f'all {endpoints.size} endpoints are nonfinite')
            elif n_nonfinite:
                warning = f'{n_nonfinite}/{endpoints.size} nonfinite endpoints'
                print(f'Warning for inset tau={tau:g}, D={D:g}: {warning}')
                inset_warnings.append({
                    'tau': tau, 'D': D, 'tau2D': D * tau**2,
                    'warning': warning,
                })
        if counts.size == 0 or counts.sum() <= 0:
            reasons.append('zero in-range histogram count')
        if reasons:
            reason = '; '.join(reasons)
            print(f'Skipping inset tau={tau:g}, D={D:g}: {reason}')
            inset_exclusions.append({
                'tau': tau, 'D': D, 'tau2D': D * tau**2, 'reason': reason,
            })
            continue
        row = summarize_result(result)
        inset_rows.append({
            'tau': tau, 'D': D, 'tau2D': D * tau**2,
            'KL': row['dkl_pooled'],
            'KL_uncertainty': row['dkl_uncertainty'],
            'outside': row['outside_total'],
            'outside_fraction': row['outside_total'] / row['samples_total'],
        })
    inset_results = pd.DataFrame(inset_rows)
    inset_exclusions = pd.DataFrame(inset_exclusions)
    inset_warnings = pd.DataFrame(inset_warnings)
    if not inset_results.empty:
        # Group on rounded log10 values: mathematically identical products can
        # differ by a few floating-point ulps when formed from different pairs.
        inset_results['tau2D_group'] = np.round(
            np.log10(inset_results['tau2D']), decimals=12
        )

    inset_curve_rows = []
    if not inset_results.empty:
        for log_tau2D, group in inset_results.groupby('tau2D_group', sort=True):
            inset_curve_rows.append({
                'tau2D': 10.0**log_tau2D,
                'log10_tau2D': log_tau2D,
                'KL': group['KL'].mean(),
                'KL_uncertainty': np.sqrt(np.square(group['KL_uncertainty']).sum()) / len(group),
                'n_pairs': len(group),
            })
    inset_curve = pd.DataFrame(inset_curve_rows)
    print(f'Valid inset configurations: {len(inset_results)}/{len(inset_stationary_data)}')
    display(inset_results.sort_values(['tau', 'D']))
    if not inset_exclusions.empty:
        print('Excluded inset configurations:')
        display(inset_exclusions.sort_values(['tau', 'D']))
    if not inset_warnings.empty:
        print('Retained configurations with partial nonfinite endpoints:')
        display(inset_warnings.sort_values(['tau', 'D']))
    display(inset_curve)


In [ ]:
clear_styles = {
    'UCNA': dict(marker='o', linestyle='-',  ms=5, lw=1.5, zorder=3),
    'Correction': dict(marker='D', linestyle=':',  ms=5, lw=1.7, zorder=4),    
    'cBFPE': dict(marker='s', linestyle='-', mfc='white', ms=6, lw=1.5, zorder=5),
}

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

if missing_inset_keys:
    print(
        f'Warning: {len(missing_inset_keys)} requested inset pairs are missing; '
        'the inset uses the available validated results.'
    )
if not inset_exclusions.empty:
    print(
        f'Warning: {len(inset_exclusions)} invalid inset configuration(s) were '
        'excluded; inspect inset_exclusions for the reasons.'
    )
if inset_results.empty:
    raise RuntimeError('No valid inset configurations are available')

styles_full = {'Correction': ':', 'UCNA': '-', 'cBFPE': '--'}
markers_full = {'Correction': 'D', 'UCNA': 'o', 'cBFPE': 's'}
taus_full = sorted(figure_2b_results['tau'].unique())
N_tau_full = len(taus_full)
colors_full = sns.color_palette(
    'viridis', n_colors=N_tau_full + 2 * (N_tau_full - 1)
)[::3]
cmap_full = mpl.colors.ListedColormap(colors_full)
norm_full = mpl.colors.BoundaryNorm(
    boundaries=np.arange(N_tau_full + 1) - 0.5, ncolors=N_tau_full
)
tau_to_idx = {tau: index for index, tau in enumerate(taus_full)}
figure_2b_results['_tau_idx_full'] = figure_2b_results['tau'].map(tau_to_idx)

fig, ax2 = plt.subplots(figsize=(5.5, 5))
for approximation in ['UCNA', 'Correction', 'cBFPE']:
    approximation_data = figure_2b_results[
        figure_2b_results['Approximation'] == approximation
    ]
    for tau, band in approximation_data.groupby('tau'):
        if approximation == 'Correction' and tau == 3: continue
        band = band.sort_values('tau2D')
        color = colors_full[tau_to_idx[tau]]
        ax2.plot(
            band['tau2D'], band['KL'], color=color, mec=color, mew=1.2,
            **clear_styles[approximation],
        )
        lower = np.maximum(
            band['KL'] - band['KL_uncertainty'], np.finfo(float).tiny
        )
        upper = band['KL'] + band['KL_uncertainty']
        ax2.fill_between(
            band['tau2D'], lower, upper, color=color, alpha=0.25,
            linewidth=0, zorder=1,
        )

invalid = figure_2b_results[(figure_2b_results['Approximation'] == 'Correction') & (~figure_2b_results['correction_valid'])]
"""for _, row in invalid.iterrows():
    ax2.scatter(
        row['tau2D'], row['KL'], marker='X', s=70,
        color=colors_full[tau_to_idx[row['tau']]], edgecolor='black',
        linewidth=0.2, zorder=5,
    )"""

ax2.hlines(
    0.5, figure_2b_results['tau2D'].min(), figure_2b_results['tau2D'].max(),
    color='grey', alpha=0.5,
)
sm = mpl.cm.ScalarMappable(cmap=cmap_full, norm=norm_full)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax2, location='top', pad=0.02, ticks=range(N_tau_full))

cbar.ax.set_xticklabels([f'$\\tau = {taus_full[0]}$'] + [f'{tau}' for tau in taus_full[1:]])
legend_handles = [
    mpl.lines.Line2D([], [], color='black', marker='o', linestyle='-',
                     mec='black', label='UCNA'),
    mpl.lines.Line2D([], [], color='black', marker='D', linestyle=':',
                     mec='black', label='Correction'),
    #mpl.lines.Line2D([], [], color='black', marker='X', linestyle='none', label='Non-positive'),
    mpl.lines.Line2D([], [], color='black', marker='s', linestyle='-',
                     mfc='white', mec='black', label='cBFPE'),
]
ax2.legend(handles=legend_handles, loc='upper left')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_yticks([1, 10, 100, 1000])
ax2.set_xlabel(r'$\mathbf{D\tau^2}$')
ax2.set_ylabel('$\mathbf{d_{KL}}$')
ax2.set_ylim(0.2, 2500)
ax2.set_xlim(0.008, 6)

axins = inset_axes(
    ax2, width='40%', height='40%', loc='lower left',
    bbox_to_anchor=(0.56, 0.05, 1, 1), bbox_transform=ax2.transAxes,
    borderpad=0.8,
)
#axins.scatter(inset_results['tau2D'], inset_results['KL'], s=12, color='tab_blue', alpha=0.22, linewidth=0)

axins.plot(inset_curve['tau2D'], inset_curve['KL'], marker='o', ms=3, color='black')
inset_lower = np.maximum(
    inset_curve['KL'] - inset_curve['KL_uncertainty'], np.finfo(float).tiny
)
axins.fill_between(
    inset_curve['tau2D'], inset_lower,
    inset_curve['KL'] + inset_curve['KL_uncertainty'],
    color='tab:blue', alpha=0.25, linewidth=0,
)
axins.set_xscale('log')
axins.set_yscale('log')
axins.yaxis.set_minor_locator(mpl.ticker.NullLocator())
axins.tick_params(labelsize=10)
axins.set_xlabel('')
axins.set_ylabel('')
ax2.annotate(r'$\left(b\right)$', (0.91, 0.9), xycoords='axes fraction',
             fontsize=15, fontweight='bold')
axins.annotate(r'$\left(c\right)$', (0.81, 0.82), xycoords='axes fraction',
               fontsize=15, fontweight='bold')
plt.show()